## Document Loading

Load PDFs and DOCX files, extract text/metadata, and save to txt files.

### Import Libraries

In [1]:
from llama_index.core import SimpleDirectoryReader
from pathlib import Path
import json

### Define Data Paths

In [2]:
# Define base paths for PDF, DOCX, and output TXT directories
BASE_DIR = Path("../data")
PDF_DIR = BASE_DIR / "pdf"
DOCX_DIR = BASE_DIR / "docx"
TXT_DIR = BASE_DIR / "txt"

# Ensure output directory exists
TXT_DIR.mkdir(parents=True, exist_ok=True)

### Load PDF Documents

> **Note:** `SimpleDirectoryReader` loads **each page as a separate document**. 
> In our example: 2 PDF files → 16 documents (pages).

In [3]:
# Load PDFs with error handling for missing directory
# Note: Each page is loaded as a separate document
try:
    pdf_documents = SimpleDirectoryReader(str(PDF_DIR), required_exts=[".pdf"]).load_data()
    pdf_files_count = len(set(doc.metadata.get('file_name', '') for doc in pdf_documents))
    print(f"Loaded {len(pdf_documents)} pages from {pdf_files_count} PDF files")
except FileNotFoundError:
    print(f"PDF directory not found: {PDF_DIR}")
    pdf_documents = []
except Exception as e:
    print(f"Error loading PDFs: {e}")
    pdf_documents = []

Loaded 16 pages from 2 PDF files


### Load DOCX Documents

In [4]:
# Load DOCXs with error handling for missing directory
try:
    docx_documents = SimpleDirectoryReader(str(DOCX_DIR), required_exts=[".docx"]).load_data()
    print(f"Loaded {len(docx_documents)} DOCX documents")
except FileNotFoundError:
    print(f"DOCX directory not found: {DOCX_DIR}")
    docx_documents = []
except Exception as e:
    print(f"Error loading DOCXs: {e}")
    docx_documents = []

Loaded 2 DOCX documents


### Explore PDF Document Metadata

In [5]:
# Print metadata of first PDF document
if pdf_documents:
    print("First PDF document metadata:")
    print(json.dumps(pdf_documents[0].metadata, indent=2))

First PDF document metadata:
{
  "page_label": "1",
  "file_name": "A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1.pdf",
  "file_path": "/Users/apple/Documents/GitHub Repositories/Lexi-Bot/notebooks/../data/pdf/A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1.pdf",
  "file_type": "application/pdf",
  "file_size": 260365,
  "creation_date": "2026-03-12",
  "last_modified_date": "2026-03-12"
}


### Explore PDF Document Text

In [6]:
# Print first 500 characters of first PDF
if pdf_documents:
    print("First 500 characters of first PDF:")
    print(pdf_documents[0].text[:500])

First 500 characters of first PDF:
A John Kennedy vs The State Of Tamil Nadu on 24 March, 2025
Author: Vikram Nath
Bench: Vikram Nath
2025 INSC 443                                                             REPORTABLE
                                      IN THE SUPREME COURT OF INDIA
                                       CIVIL APPELLATE JURISDICTION
                                      SLP(CIVIL) NO(S). 999-1001 OF 2025
            A. JOHN KENNEDY ETC.                                          …PETITIONER(S)
                  


### Explore DOCX Document Metadata

In [7]:
# Print metadata of first DOCX document
if docx_documents:
    print("First DOCX document metadata:")
    print(json.dumps(docx_documents[0].metadata, indent=2))

First DOCX document metadata:
{
  "file_name": "A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1.docx",
  "file_path": "/Users/apple/Documents/GitHub Repositories/Lexi-Bot/notebooks/../data/docx/A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1.docx",
  "file_type": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
  "file_size": 22623,
  "creation_date": "2026-03-12",
  "last_modified_date": "2026-03-12"
}


### Save Extracted PDF Text to Files

In [8]:
# Save each PDF's text to a txt file with _extracted_from_pdf.txt suffix
# Group pages by filename and combine text
from collections import defaultdict

pdf_text_by_file = defaultdict(str)
for doc in pdf_documents:
    file_name = doc.metadata.get('file_name', 'unknown')
    pdf_text_by_file[file_name] += doc.text + "\n\n"

for file_name, combined_text in pdf_text_by_file.items():
    base_name = Path(file_name).stem  # filename without extension
    output_file = TXT_DIR / f"{base_name}_extracted_from_pdf.txt"
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(combined_text)
    
    print(f"Saved: {output_file}")

Saved: ../data/txt/A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_pdf.txt
Saved: ../data/txt/A_Rajendra_vs_Gonugunta_Madhusudhan_Rao_on_4_April_2025_1_extracted_from_pdf.txt


### Save Extracted DOCX Text to Files

In [9]:
# Save each DOCX's text to a txt file with _extracted_from_docx.txt suffix
for doc in docx_documents:
    file_name = doc.metadata.get('file_name', 'unknown')
    base_name = Path(file_name).stem  # filename without extension
    output_file = TXT_DIR / f"{base_name}_extracted_from_docx.txt"
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(doc.text)
    
    print(f"Saved: {output_file}")

Saved: ../data/txt/A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt
Saved: ../data/txt/A_Rajendra_vs_Gonugunta_Madhusudhan_Rao_on_4_April_2025_1_extracted_from_docx.txt


### Summary Statistics

In [10]:
# Print summary of all loaded documents
all_docs = pdf_documents + docx_documents

pdf_files_count = len(set(doc.metadata.get('file_name', '') for doc in pdf_documents))
pdf_pages_count = len(pdf_documents)
pdf_chars = sum(len(doc.text) for doc in pdf_documents)
pdf_unique_pages = len(set(doc.metadata.get('page_label', '1') for doc in pdf_documents))

docx_files_count = len(set(doc.metadata.get('file_name', '') for doc in docx_documents))
docx_docs_count = len(docx_documents)
docx_chars = sum(len(doc.text) for doc in docx_documents)

print("=== Document Summary ===")
print(f"\nPDFs:")
print(f"  Files: {pdf_files_count}")
print(f"  Pages loaded: {pdf_pages_count}")
print(f"  Unique pages: {pdf_unique_pages}")
print(f"  Characters: {pdf_chars:,}")
print(f"\nDOCXs:")
print(f"  Files: {docx_files_count}")
print(f"  Documents: {docx_docs_count}")
print(f"  Characters: {docx_chars:,}")
print(f"\nTotal:")
print(f"  Total files: {pdf_files_count + docx_files_count}")
print(f"  Total documents/pages: {len(all_docs):,}")
print(f"  Total characters: {pdf_chars + docx_chars:,}")

=== Document Summary ===

PDFs:
  Files: 2
  Pages loaded: 16
  Unique pages: 9
  Characters: 48,022

DOCXs:
  Files: 2
  Documents: 2
  Characters: 45,865

Total:
  Total files: 4
  Total documents/pages: 18
  Total characters: 93,887


### List All Extracted Files

In [11]:
# List all txt files created in the output directory
txt_files = list(TXT_DIR.glob("*.txt"))
print(f"Extracted files in {TXT_DIR}:")
for f in sorted(txt_files):
    print(f"  - {f.name}")

Extracted files in ../data/txt:
  - A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt
  - A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_pdf.txt
  - A_Rajendra_vs_Gonugunta_Madhusudhan_Rao_on_4_April_2025_1_extracted_from_docx.txt
  - A_Rajendra_vs_Gonugunta_Madhusudhan_Rao_on_4_April_2025_1_extracted_from_pdf.txt
